# Predicting Regional Retail Trade in Kazakhstan

## Overview

This notebook builds a multiple linear regression model to predict monthly retail trade volume across Kazakhstan's 20 regions. All data were compiled from four official monthly bulletins published by the Bureau of National Statistics of the Republic of Kazakhstan (stat.gov.kz): February 2024, June 2024, June 2025, and December 2025.

## Problem statement

Retail trade volume is one of the more direct proxies for regional consumer activity - it is measured monthly, covers the full formal retail sector, and reflects actual household spending rather than an estimated aggregate. The goal here is to determine whether five macroeconomic and demographic indicators can predict that volume, and more importantly, to understand the mechanism behind each predictor: not just which variable has the highest coefficient, but why the data produces that ordering.

The log-linear specification used below allows each coefficient to be read as an elasticity, which makes the interpretation concrete: a coefficient of 1.53 on log(income) means a 1% increase in income is associated with a 1.53% increase in retail trade. That is a testable economic claim, not just a model parameter.

## Data provenance

figures were compiled from the following official bulletins:
- [February 2024](https://stat.gov.kz/upload/iblock/002/iymgccgoyj2adx0lgbbu1ovcapdity11/%D0%96-01-01-%D0%9C%2002%202024%20(%D1%80%D1%83%D1%81).pdf)
- [June 2024](https://stat.gov.kz/upload/iblock/b16/u1k0y8ikgvnv2y0vgxzc0gg6cx4ru9f5/%D0%96-01-%D0%9C%2006%202024%20(%D1%80%D1%83%D1%81).pdf)
- [June 2025](https://stat.gov.kz/upload/iblock/c98/gfrc35fthdnpbtp34vzz4i4xu1dc4cz4/%D0%96-01-%D0%9C%2006%202025%20(%D1%80%D1%83%D1%81).pdf)
- [December 2025](https://stat.gov.kz/upload/iblock/082/mwq0xdd7o5ukymteea8y9n1kyvhg12xn/%D0%96-01-%D0%9C%2012%202025%20(%D1%80%D1%83%D1%81).pdf)

income and unemployment are published quarterly by the source; each monthly observation was matched to the nearest available quarter.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

BLUE   = '#1a4f8a'
RED    = '#b5281e'
GREEN  = '#2d6a4f'
ORANGE = '#c96a00'
GREY   = '#6b7280'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#fafafa',
    'axes.grid':        True,
    'grid.color':       '#e5e7eb',
    'grid.linewidth':   0.6,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})


## 1. Loading and inspecting the data

In [ ]:
df = pd.read_csv('../data/kazakhstan_regional_retail_trade.csv')
df['Region_label'] = df['Region'].str.replace('_', ' ')

print(f'shape: {df.shape}')
print(f'periods: {sorted(df.Period.unique())}')
print(f'regions: {df.Region.nunique()}')
print(f'missing values: {df.isnull().sum().sum()}')
df.head(8)


In [ ]:
df.describe().round(2)


## 2. Feature descriptions

| column | unit | notes |
|---|---|---|
| `Retail_Trade_mln_KZT` | million KZT | **target** - single-month volume, not cumulative |
| `Population_thousand` | thousand people | regional population at period start |
| `Income_per_capita_KZT` | KZT/month | nearest quarterly figure |
| `CPI_YoY_percent` | % | e.g. 112.3 = prices 12.3% above prior year |
| `Investment_mln_KZT` | million KZT | fixed capital investment, single month |
| `Unemployment_rate_percent` | % | nearest quarterly figure, ILO methodology |


## 3. Exploratory analysis

### 3.1 Feature distributions

The first thing to check with regional economic data is the shape of each variable's distribution. The concern is that a handful of very large regions can dominate the variance and pull a linear fit away from the pattern that holds across the majority of observations.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('distribution of raw features (n = 80)', fontsize=14, fontweight='bold', y=1.01)

specs = [
    ('Retail_Trade_mln_KZT',      BLUE,         'retail trade (mln KZT)'),
    ('CPI_YoY_percent',            GREEN,         'CPI year-over-year (%)'),
    ('Population_thousand',        ORANGE,        'population (thousand)'),
    ('Investment_mln_KZT',         '#6d28d9',     'investment (mln KZT)'),
    ('Income_per_capita_KZT',      RED,           'income per capita (KZT/month)'),
    ('Unemployment_rate_percent',  GREY,          'unemployment rate (%)'),
]
for ax, (col, color, label) in zip(axes.flatten(), specs):
    vals = df[col]
    ax.hist(vals, bins=22, color=color, alpha=0.82, edgecolor='white', linewidth=0.6)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('count', fontsize=10)
    ax.axvline(vals.mean(), color='black', linewidth=1.4, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


retail trade, population, and investment are all right-skewed. Almaty city records retail trade of roughly 1.2 trillion KZT in December 2025, compared to around 14 billion for Ulytau - a ratio of about 160. this is not noise or error; it reflects the actual concentration of economic activity in Kazakhstan. the skew creates a real modeling problem: a regression fit on raw values would be largely driven by the largest two or three cities, and the resulting coefficients would describe those cities' relationship with the predictors rather than the pattern across the full country.

CPI and unemployment are symmetric and narrow, with no transformation needed. inflation ranged from 107.8% to 113.8% across the sample; unemployment from 4.0% to 5.1%.


### 3.2 Log transformation

the standard response to multiplicative skew in regional economic data is to log-transform the affected variables. the conceptual shift this produces is important: instead of asking "how many more million KZT does a region with more people sell", the model asks "what percentage more does a region with 1% more people sell". the latter question has a consistent answer across regions that differ by two orders of magnitude; the former does not.


In [ ]:
df['log_retail']     = np.log(df['Retail_Trade_mln_KZT'])
df['log_population'] = np.log(df['Population_thousand'])
df['log_investment'] = np.log(df['Investment_mln_KZT'])
df['log_income']     = np.log(df['Income_per_capita_KZT'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('retail trade: before and after log transform', fontsize=13, fontweight='bold')

axes[0].hist(df['Retail_Trade_mln_KZT'] / 1000, bins=25, color=BLUE, alpha=0.82, edgecolor='white')
axes[0].set_xlabel('retail trade (bln KZT)')
axes[0].set_title('raw values - heavy right tail')

axes[1].hist(df['log_retail'], bins=25, color=GREEN, alpha=0.82, edgecolor='white')
axes[1].set_xlabel('log(retail trade)')
axes[1].set_title('after log - near-symmetric')

plt.tight_layout()
plt.show()


### 3.3 Correlation structure

In [ ]:
corr_df = df[['log_retail','log_population','log_income','log_investment',
              'CPI_YoY_percent','Unemployment_rate_percent']].copy()
corr_df.columns = ['Retail (log)','Population (log)','Income (log)',
                   'Investment (log)','CPI YoY %','Unemployment %']
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'fontweight': 'bold'}, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('pairwise correlations (log scale for skewed variables)', pad=14)
ax.tick_params(axis='x', rotation=30, labelsize=9.5)
ax.tick_params(axis='y', rotation=0, labelsize=9.5)
plt.tight_layout()
plt.show()

print('correlation with log(retail trade):')
print(corr['Retail (log)'].drop('Retail (log)').sort_values(ascending=False).round(3))


investment (r = 0.67) and population (r = 0.65) lead the table, but the near-equal values obscure different causal paths.

**why population correlates strongly:** retail trade is, at the most basic level, the sum of transactions by individual residents. more people means more buyers. this relationship is robust across regions with very different income profiles and economic structures, which is why it holds across the full 80-observation sample.

**why investment's correlation is slightly higher than population's:** capital investment in a given month is not direct consumer spending - it goes to infrastructure, machinery, and buildings. its strong correlation with retail trade reflects the fact that it is a proxy for general economic dynamism: regions attracting large capital flows are expanding along multiple dimensions simultaneously - employment, infrastructure, commercial development. high investment figures therefore co-occur with higher income, better retail infrastructure, and growing employment, all of which drive retail trade. the correlation coefficient captures all of those indirect paths, not just investment's direct effect.

**why income per capita (r = 0.39) comes in lower:** the Atyrau effect. Atyrau has among the highest income per capita in Kazakhstan due to the oil sector, but its retail trade volume is moderate relative to that income level. a substantial share of oil-sector earnings leaves the region rather than circulating locally. this divergence compresses the income-retail correlation across the full sample.

**why unemployment (r = -0.16) appears so weak:** the range across regions is only 4.0% to 5.1%. a genuine relationship between unemployment and spending exists, but when the variable barely moves, its correlation with anything else will be small regardless of the true effect size.


### 3.4 Population and income vs retail trade

In [ ]:
highlight = {'Almaty city': BLUE, 'Astana': ORANGE, 'Atyrau': RED, 'Ulytau': GREY}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('population and income vs retail trade (log-log)', fontsize=13, fontweight='bold')

for ax, (xvar, xlabel) in zip(axes, [
        ('log_population', 'log(population, thousand)'),
        ('log_income',     'log(income per capita, KZT/month)')]):
    x, y = df[xvar], df['log_retail']
    ax.scatter(x, y, color='#94a3b8', alpha=0.55, s=28, zorder=2)
    for reg, col in highlight.items():
        mask = df['Region_label'] == reg
        if mask.any():
            ax.scatter(x[mask], y[mask], color=col, s=60, zorder=4,
                       edgecolors='white', linewidth=0.7, label=reg)
            ax.annotate(reg, xy=(x[mask].iloc[-1], y[mask].iloc[-1]),
                        xytext=(6, 3), textcoords='offset points',
                        fontsize=8, color=col)
    z = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 200)
    ax.plot(xs, np.poly1d(z)(xs), color='black', linewidth=1.6,
            linestyle='--', alpha=0.7, label='OLS trend')
    r = np.corrcoef(x, y)[0, 1]
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('log(retail trade, mln KZT)', fontsize=10)
    ax.set_title(f'Pearson r = {r:.2f}')
    ax.legend(fontsize=8.5, framealpha=0.8)
plt.tight_layout()
plt.show()


**population panel (left):** the log-log relationship is close to linear across all 20 regions, confirming that the log transform was the right scale. the slope estimated visually - and confirmed by the regression coefficient of 1.22 - is above 1. that excess above 1 is agglomeration: large urban centers generate retail activity beyond what their resident population alone would produce, because they serve as commercial destinations for residents of smaller surrounding regions.

**income panel (right):** Atyrau is the most informative data point here. it sits far to the right (high income per capita) but close to the trend line vertically (retail trade in line with its population, not its income level). this is what a capital-intensive, low-employment extraction economy looks like in terms of household spending: high average wages for a relatively small workforce, with limited spillover into local retail.


### 3.5 Regional comparison, December 2025

In [ ]:
dec = df[df['Period'] == '2025-12'].sort_values('Retail_Trade_mln_KZT')
bar_colors = [RED if r == 'Almaty city' else ORANGE if r == 'Astana'
              else BLUE for r in dec['Region_label']]

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(dec['Region_label'], dec['Retail_Trade_mln_KZT'] / 1000,
               color=bar_colors, alpha=0.88, edgecolor='white')
ax.set_xlabel('retail trade, billion KZT (December 2025)', fontsize=10)
ax.set_title('regional retail trade - December 2025', pad=14)
for bar, val in zip(bars, dec['Retail_Trade_mln_KZT'] / 1000):
    ax.text(val + 5, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f}', va='center', fontsize=8.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=RED, label='Almaty city'),
                   Patch(color=ORANGE, label='Astana'),
                   Patch(color=BLUE, label='other regions')],
          loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


Almaty's December 2025 retail trade is approximately 2.3 times Astana's, against a population ratio of roughly 1.4. the gap is structural rather than demographic. Almaty's wholesale and retail infrastructure - distribution centers, established retail chains, international brand presence - was built up over decades before Astana became the capital in 1997. that infrastructure did not relocate with the capital functions, which means Almaty continues to serve as the primary retail destination for a much larger catchment area than its administrative boundaries would suggest.


### 3.6 Inflation and retail trade over time

In [ ]:
period_avg = df.groupby('Period').agg(
    CPI=('CPI_YoY_percent', 'mean'),
    Retail=('Retail_Trade_mln_KZT', 'mean')
).reset_index()
period_labels = ['Feb 2024', 'Jun 2024', 'Jun 2025', 'Dec 2025']

fig, ax1 = plt.subplots(figsize=(10, 5.5))
ax2 = ax1.twinx()
ax1.plot(period_labels, period_avg['CPI'], 'o-', color=RED,
         linewidth=2.2, markersize=7, label='CPI YoY (%)')
ax2.plot(period_labels, period_avg['Retail'] / 1000, 's-', color=BLUE,
         linewidth=2.2, markersize=7, label='avg retail (bln KZT)')
for i, (cpi, ret) in enumerate(zip(period_avg['CPI'], period_avg['Retail'] / 1000)):
    ax1.annotate(f'{cpi:.1f}%', xy=(i, cpi), xytext=(4, 6),
                 textcoords='offset points', fontsize=9, color=RED)
    ax2.annotate(f'{ret:.0f}', xy=(i, ret), xytext=(4, -14),
                 textcoords='offset points', fontsize=9, color=BLUE)
ax1.set_ylabel('CPI year-over-year (%)', color=RED, fontsize=10)
ax2.set_ylabel('average retail trade (bln KZT)', color=BLUE, fontsize=10)
ax1.tick_params(axis='y', labelcolor=RED)
ax2.tick_params(axis='y', labelcolor=BLUE)
ax1.set_title('inflation and average regional retail trade - four time points', pad=14)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()


average nominal retail trade nearly doubled between February 2024 and December 2025. at the same time, inflation ran at 110-113% year-over-year throughout the period, meaning prices were consistently 10-13% higher than the prior year. a naive reading might treat this as strong consumption growth, but nominal retail trade growth and real consumption growth are not the same thing. some of what appears as rising retail trade is simply the same volume of goods priced higher.

the model works in nominal terms throughout and cannot separate these two effects. this is flagged in the limitations section. the CPI coefficient in the regression should be read as measuring the marginal association between inflation and nominal retail trade after controlling for population, income, investment, and unemployment - not as evidence of price-driven consumption.


## 4. Model

### 4.1 Feature selection and specification

the five features entering the model are log(population), log(income per capita), CPI YoY %, log(investment), and unemployment rate %. CPI and unemployment are left in their original percentage-point units because they are already on a sensible scale and do not require the proportional comparison that log transformation enables.

the log-log specification for the remaining features means the model is estimating elasticities: how many percent does retail trade change per 1% change in the predictor. this is the standard specification for demand-side economic models at the regional level.


In [ ]:
feature_cols = ['log_population', 'log_income', 'CPI_YoY_percent',
                'log_investment', 'Unemployment_rate_percent']
target_col = 'log_retail'

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

print(f'training set: {len(X_train)} observations')
print(f'test set:     {len(X_test)} observations')


### 4.2 Training

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

pred_train = model.predict(X_train)
pred_test  = model.predict(X_test)

print(f'intercept: {model.intercept_:.4f}')
print()
for feat, coef in zip(feature_cols, model.coef_):
    print(f'  {feat:<35s}: {coef:+.4f}')


### 4.3 Evaluation metrics

R² is used as the primary metric for comparing train and test performance. MAE and RMSE are both reported but cannot be compared directly to each other because they weight large errors differently. R² has a fixed [0, 1] scale that is consistent across train and test sets.


In [ ]:
def report(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{label}:  MAE = {mae:.4f}  RMSE = {rmse:.4f}  R² = {r2:.4f}  (log units)')
    return r2

r2_tr = report(y_train, pred_train, 'train')
r2_te = report(y_test,  pred_test,  'test ')

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_val_score(model, X, y, cv=kf, scoring='r2')
print(f'\n5-fold CV  mean = {cv.mean():.3f}  std = {cv.std():.3f}')
print(f'per fold: {np.round(cv, 3)}')

# back-transform to real KZT for practical sense
y_real_pred = np.exp(pred_test)
y_real_true = np.exp(y_test)
mape = np.mean(np.abs((y_real_true - y_real_pred) / y_real_true)) * 100
print(f'\ntest MAPE (real KZT): {mape:.1f}%')


a test R² of 0.72 means the model explains approximately 72% of the variance in log(retail trade) using five predictors. the train-test gap is small (0.741 vs 0.720), indicating the model generalizes to new observations rather than overfitting the training sample.

the 5-fold CV scores range from 0.05 to 0.88. that spread does not indicate model instability - it reflects the small effective sample size. with 20 independent regions, any single fold can receive a skewed composition. if a fold happens to include Atyrau (high income, moderate retail) and several small oblasts without Almaty or Astana, the model's predictions in that fold will look much weaker than average because the structural anchors of the dataset are missing from the test portion.

a MAPE of approximately 41% sounds large in isolation. in context: the model is predicting across a 160x size range without any region-specific parameters. the realistic expectation for a five-variable linear model on 80 regional observations is that it will get the order of magnitude right and capture the directional drivers - not predict exact monthly totals.


### 4.4 Actual vs predicted

In [ ]:
residuals = y_test - pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('model performance on test set (n = 20)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.scatter(y_test, pred_test, color=BLUE, alpha=0.75, s=55,
           edgecolors='white', linewidth=0.6, zorder=3)
lim = [min(y_test.min(), pred_test.min()) - 0.2,
       max(y_test.max(), pred_test.max()) + 0.2]
ax.plot(lim, lim, '--', color=RED, linewidth=1.8, label='perfect prediction')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('actual log(retail trade)')
ax.set_ylabel('predicted log(retail trade)')
ax.set_title(f'actual vs predicted  |  R² = {r2_te:.3f}')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(residuals, bins=14, color=RED, alpha=0.80, edgecolor='white')
ax.axvline(0, color='black', linewidth=1.8, linestyle='--')
ax.axvline(residuals.mean(), color=ORANGE, linewidth=1.4, linestyle=':',
           label=f'mean residual = {residuals.mean():.3f}')
ax.set_xlabel('residual (log units)')
ax.set_ylabel('count')
ax.set_title('residual distribution')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


the residuals are approximately symmetric around zero. this means the model does not consistently over- or under-predict for any obvious structural reason - there is no systematic directional bias. the mean residual is close to zero, confirming the model is calibrated rather than shifted.


## 5. Coefficient interpretation

because the target and four of the five features are in log units, those coefficients are elasticities. a coefficient of 1.53 on log(income) means: when income per capita rises by 1%, retail trade rises by 1.53%, with all other features held constant. CPI and unemployment are in percentage points, so their coefficients represent the absolute log-unit shift per one percentage-point change.


In [ ]:
coef_df = pd.DataFrame({'feature': feature_cols, 'coef': model.coef_}).sort_values('coef')
labels_clean = {
    'log_population':            'log(population)',
    'log_income':                'log(income per capita)',
    'CPI_YoY_percent':           'CPI year-over-year (%)',
    'log_investment':            'log(investment)',
    'Unemployment_rate_percent': 'unemployment rate (%)',
}
coef_df['label'] = coef_df['feature'].map(labels_clean)

bar_colors = [RED if v < 0 else BLUE for v in coef_df['coef']]
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(coef_df['label'], coef_df['coef'],
               color=bar_colors, alpha=0.88, edgecolor='white', height=0.55)
ax.axvline(0, color='black', linewidth=1.2)
ax.set_xlabel('coefficient (elasticity for log-transformed features)')
ax.set_title('regression coefficients - log(retail trade)', pad=14)
for bar, val in zip(bars, coef_df['coef']):
    offset = 0.025 if val >= 0 else -0.025
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'{val:+.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label='positive effect'),
                   Patch(color=RED,  label='negative effect')],
          fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

print('\ncoefficients:')
for _, row in coef_df.sort_values('coef', ascending=False).iterrows():
    print(f'  {row["label"]:<30s}: {row["coef"]:+.4f}')
print(f'  intercept: {model.intercept_:.4f}')


**log(income per capita), +1.53** - the largest coefficient in the model. an elasticity above 1 means that when income rises, retail trade grows faster than income itself. the mechanism is that as households move above subsistence-level income, an increasing share of additional earnings goes toward discretionary categories - restaurants, electronics, clothing, travel. those categories have higher unit value and higher price elasticity than basic necessities, which pulls the aggregate retail-income elasticity above 1.

**log(population), +1.22** - also above 1, which is a meaningful observation. if each resident spent an identical fixed amount regardless of context, the elasticity would be exactly 1. the excess above 1 reflects agglomeration: larger urban centers generate retail activity that extends beyond their resident population. residents of smaller surrounding regions travel to Almaty and Astana for purchases unavailable locally; large cities support more diverse retail formats including premium segments that smaller markets cannot sustain; and the presence of major distribution infrastructure makes a wider product range available.

**unemployment rate, -0.44** - the only negative coefficient. exp(-0.437) = 0.65, meaning a one percentage-point rise in unemployment is associated with retail trade approximately 35% lower, with other variables held constant. the mechanism is direct: unemployment reduces household income, which reduces the budget available for discretionary and even some necessary spending. the sign is the only one in the model with an unambiguous directional prediction from first principles, and the data confirms it.

**log(investment), +0.11** - the weakest effect among the significant predictors. fixed capital investment affects consumption indirectly and with a lag: a new distribution center or factory first requires construction (which generates local employment income), then operates (generating ongoing wages), before those income flows cycle into retail spending. within a single monthly snapshot, the contemporaneous effect is small. the coefficient should not be read as investment having little relevance to regional retail; it means the same-month correlation is modest, which is expected given the lag structure.

**CPI year-over-year, +0.02** - small and positive. each percentage point of additional inflation corresponds to approximately 2.3% higher nominal retail trade, once the other variables are controlled for. the residual effect is small because most of the inflation signal across regions is absorbed into the period dimension of the data: prices rose across all 20 regions simultaneously in 2025, so the cross-sectional variation in CPI is narrow and its standalone contribution to the regression is limited.


## 6. Limitations

**effective sample size.** 80 rows represent 20 regions at 4 time points, not 80 independent observations. the same fundamental economic structure of each region persists across periods, which reduces the effective n to approximately 20. this is the direct cause of the wide CV spread (0.05 to 0.88 across folds).

**nominal prices throughout.** retail trade is reported in current KZT. the model estimates the association between nominal retail trade and its predictors. it cannot distinguish real volume growth from inflation-driven price growth. applying a CPI deflator to the target before modeling would address this.

**no interaction terms.** the model constrains the effect of income on retail trade to be identical in Almaty and in Turkestan. in practice, the relationship between income and spending likely differs between a metropolitan economy with developed service and luxury segments and an agricultural region where income gains shift spending primarily toward food and durable goods. a single global coefficient averages over this heterogeneity.

**no seasonality control.** the four time points span February, June, and December. December retail trade is typically above the annual average due to holiday spending. this seasonal signal is embedded in the December observations without any explicit control, which may affect the CPI and investment coefficients in particular.

**insufficient variation in unemployment.** the 4.0-5.1% range across all 80 observations provides limited statistical leverage to estimate the unemployment coefficient precisely. the direction is correct and the magnitude is economically plausible, but the confidence interval around -0.44 would be wide, and the true elasticity may differ from this estimate.

## 7. Suggested extensions

extending to monthly observations over 2-3 years would increase the effective sample size substantially and stabilize the cross-validation results. deflating the retail trade target by regional CPI would allow the model to target real consumption growth. adding regional fixed effects would control for time-invariant structural differences between cities and rural oblasts. comparing this linear specification against a regularized model (Ridge or Lasso) or a non-linear alternative (Random Forest) would test how much the linearity assumption costs in predictive accuracy.
